# Lattice ABP Steady-State Sanity CheckThis notebook verifies that the Lattice ABP simulation reaches a propersteady state before training the Corr model. Checks include:- Density time series- Jammed fraction evolution- Autocorrelation analysis- Convergence test (last 10% vs overall mean)

In [ ]:
import sys, osCNEEP_V2_ROOT = os.path.abspath('/home/user1/CNEEP_v2')if CNEEP_V2_ROOT not in sys.path:    sys.path.append(CNEEP_V2_ROOT)sys.path.append(os.path.join(CNEEP_V2_ROOT, 'data'))import numpy as npimport torchfrom tqdm import tqdmimport matplotlib.pyplot as pltfrom data.lattice_abp.core import LatticeABP

## 1. Configuration

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"abp_params = dict(    L=64,    v_plus=1.0,    v_zero=0.1,    v_minus=0.01,    D_rot=0.1,    density=0.5,    bc_mode='periodic',    device=device,    seed=42,)# Long simulation for steady-state analysisn_steps     = 50000burn_in     = 0       # No burn-in: we want to see the transientsave_interval = 10tau         = 0.01B           = 4       # Small ensemble for diagnosticsoutput_dir = os.path.join(CNEEP_V2_ROOT, 'results', 'sanity_lattice_abp')os.makedirs(output_dir, exist_ok=True)print(f"Device: {device}")print(f"Output: {output_dir}")

## 2. Run Long Simulation

In [ ]:
sim = LatticeABP(**abp_params)print(f"Running {n_steps} steps (tau={tau}, save_interval={save_interval})...")result = sim.simulate(    B=B,    n_steps=n_steps,    burn_in=burn_in,    method='tau_leap',    tau=tau,    save_interval=save_interval,    show_progress=True,)O_traj = result['O_traj']  # (n_saved, B, L, L)E_traj = result['E_traj']times  = result['times']   # (n_saved, B)n_saved = O_traj.shape[0]print(f"Saved {n_saved} snapshots, shape: {O_traj.shape}")

## 3. Density Time Series

In [ ]:
## Mean occupancy (density) per snapshot — should converge to abp_params['density']#L = abp_params['L']density_ts = O_traj.float().mean(dim=(-2, -1))  # (n_saved, B)density_mean = density_ts.mean(dim=1).numpy()     # (n_saved,)density_std  = density_ts.std(dim=1).numpy()time_axis = np.arange(n_saved) * save_interval * taufig, ax = plt.subplots(figsize=(12, 4))ax.plot(time_axis, density_mean, lw=1, color='steelblue', label='Mean density')ax.fill_between(time_axis,                density_mean - density_std,                density_mean + density_std,                alpha=0.2, color='steelblue')ax.axhline(abp_params['density'], color='crimson', ls='--', lw=1.5, label=f"Target ρ={abp_params['density']}")ax.set_xlabel('Simulation Time')ax.set_ylabel('Mean Occupancy')ax.set_title('Density Conservation Check')ax.legend()ax.grid(alpha=0.3)plt.tight_layout()plt.savefig(f'{output_dir}/density_timeseries.png', dpi=150)plt.show()print(f"Final density: {density_mean[-1]:.6f} (target: {abp_params['density']})")

## 4. Jammed Fraction Evolution

In [ ]:
## Track jammed fraction over time#print("Computing jammed fraction over time...")jammed_frac = []for t in tqdm(range(0, n_saved, max(1, n_saved // 500))):    O_t = O_traj[t].to(device)    E_t = E_traj[t].to(device)    jammed = sim.compute_jammed_mask(O_t, E_t)    n_particles = O_t.sum(dim=(-2, -1)).float()    n_jammed = (jammed & (O_t == 1)).sum(dim=(-2, -1)).float()    frac = (n_jammed / n_particles.clamp(min=1)).mean().item()    jammed_frac.append(frac)jf_time = np.linspace(0, time_axis[-1], len(jammed_frac))fig, ax = plt.subplots(figsize=(12, 4))ax.plot(jf_time, jammed_frac, lw=1, color='#e74c3c')ax.fill_between(jf_time, jammed_frac, alpha=0.15, color='#e74c3c')ax.set_xlabel('Simulation Time')ax.set_ylabel('Jammed Fraction')ax.set_title('Jammed Fraction Evolution')ax.grid(alpha=0.3)plt.tight_layout()plt.savefig(f'{output_dir}/jammed_fraction.png', dpi=150)plt.show()

## 5. Autocorrelation Analysis

In [ ]:
## Autocorrelation of the jammed fraction time series#def autocorrelation_fft(x, max_lag=None):    x = np.asarray(x, dtype=float)    n = len(x)    if max_lag is None:        max_lag = n // 2    x_centered = x - np.mean(x)    f = np.fft.fft(x_centered, 2 * n)    acf = np.fft.ifft(f * np.conj(f)).real[:n]    acf /= acf[0] if acf[0] != 0 else 1    return acf[:max_lag]jf_arr = np.array(jammed_frac)acf = autocorrelation_fft(jf_arr, max_lag=len(jf_arr) // 2)# Estimate autocorrelation time (first zero crossing)try:    tau_corr_idx = np.where(acf < 0)[0][0]except IndexError:    tau_corr_idx = len(acf)tau_corr = tau_corr_idx * (jf_time[1] - jf_time[0])fig, ax = plt.subplots(figsize=(12, 4))lag_time = np.arange(len(acf)) * (jf_time[1] - jf_time[0])ax.plot(lag_time, acf, lw=1.5, color='darkorange')ax.axhline(0, color='gray', ls='--', lw=0.8)ax.axvline(tau_corr, color='crimson', ls='--', lw=1.5, label=f'τ_corr ≈ {tau_corr:.2f}')ax.set_xlabel('Lag Time')ax.set_ylabel('Autocorrelation')ax.set_title('Jammed Fraction Autocorrelation')ax.legend()ax.grid(alpha=0.3)plt.tight_layout()plt.savefig(f'{output_dir}/autocorrelation.png', dpi=150)plt.show()print(f"Estimated autocorrelation time: {tau_corr:.4f}")

## 6. Convergence Test (Steady-State Check)

In [ ]:
## Compare last 10% window mean to overall mean#def steady_state_check(obs, tolerance=0.01, window_frac=0.1):    obs = np.asarray(obs)    overall_mean = np.mean(obs)    window_len = int(len(obs) * window_frac)    final_mean = np.mean(obs[-window_len:])    rel_diff = np.abs(final_mean - overall_mean) / (np.abs(overall_mean) + 1e-12)    return rel_diff < tolerance, rel_diff, overall_mean, final_mean# Check densitydensity_arr = density_meand_pass, d_diff, d_overall, d_final = steady_state_check(density_arr)# Check jammed fractionj_pass, j_diff, j_overall, j_final = steady_state_check(jf_arr)# EPR checkepr_ts = []for t in tqdm(range(0, n_saved, max(1, n_saved // 200))):    O_t = O_traj[t].to(device)    E_t = E_traj[t].to(device)    epr = sim.compute_local_epr(O_t, E_t).sum(dim=(-2,-1)).mean().item()    epr_ts.append(epr)epr_arr = np.array(epr_ts)e_pass, e_diff, e_overall, e_final = steady_state_check(epr_arr)# Summary tableimport pandas as pddf = pd.DataFrame({    'Observable': ['Density', 'Jammed Fraction', 'Total EPR'],    'Overall Mean': [d_overall, j_overall, e_overall],    'Last 10% Mean': [d_final, j_final, e_final],    'Rel. Diff (%)': [d_diff*100, j_diff*100, e_diff*100],    'Converged (<1%)': ['✅' if d_pass else '❌',                        '✅' if j_pass else '❌',                        '✅' if e_pass else '❌'],})print(df.to_string(index=False))print()if d_pass and j_pass and e_pass:    print("✅ All observables have converged — system is in steady state.")else:    print("⚠️ Some observables have NOT converged. Consider longer burn-in or more steps.")

## 7. Snapshots (Start / Middle / End)

In [ ]:
## Density snapshots at three time points#indices = [0, n_saved // 2, n_saved - 1]fig, axes = plt.subplots(1, 3, figsize=(15, 4))for ax, idx in zip(axes, indices):    occ = O_traj[idx, 0].cpu().numpy().astype(float)    im = ax.imshow(occ, cmap='Blues', interpolation='nearest', vmin=0, vmax=1)    ax.set_title(f't = {idx * save_interval * tau:.1f}')    ax.set_xticks([]); ax.set_yticks([])fig.suptitle('Occupancy Snapshots', fontsize=14)plt.tight_layout()plt.savefig(f'{output_dir}/snapshots.png', dpi=150)plt.show()

## 8. EPR Time Series

In [ ]:
## EPR time series to verify convergence#epr_time = np.linspace(0, time_axis[-1], len(epr_ts))fig, axes = plt.subplots(2, 1, figsize=(12, 7))axes[0].plot(epr_time, epr_arr, lw=0.8, color='steelblue', alpha=0.7)# Running averageif len(epr_arr) > 20:    w = min(50, len(epr_arr) // 4)    epr_smooth = np.convolve(epr_arr, np.ones(w)/w, mode='same')    axes[0].plot(epr_time, epr_smooth, lw=2, color='crimson', label=f'Running avg (w={w})')axes[0].set_ylabel('Total EPR')axes[0].set_title('EPR Time Series')axes[0].legend()axes[0].grid(alpha=0.3)# EPR map at final timeO_final = O_traj[-1].to(device)E_final = E_traj[-1].to(device)epr_map_final = sim.compute_local_epr(O_final, E_final).mean(dim=0).cpu().numpy()im = axes[1].imshow(epr_map_final.T, origin='lower', cmap='hot')axes[1].set_title('Final EPR Density Map')fig.colorbar(im, ax=axes[1])plt.tight_layout()plt.savefig(f'{output_dir}/epr_diagnostics.png', dpi=150)plt.show()print(f"Final total EPR: {epr_arr[-1]:.6e}")